In [ ]:
import matplotlib.pyplot as mp
import numpy as np
import scipy
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

import pyfauxseq as pf

In [ ]:
seed = np.random.randint(low=0, high=1e7)

### RNA-seq data under one condition

In [ ]:
sim_1c_data = pf.generate_rhythmic_rnaseq(reps=3, n_genes=10000, rhy_frac=0.1, seed=seed)
sim_1c_data["exp_design"]["inphase"] = np.cos(2*np.pi/24*sim_1c_data["exp_design"].time.values)
sim_1c_data["exp_design"]["outphase"] = np.sin(2*np.pi/24*sim_1c_data["exp_design"].time.values)

In [ ]:
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(counts = sim_1c_data["counts"].T, metadata=sim_1c_data["exp_design"], design="~inphase + outphase", refit_cooks=True, inference=inference)
dds.deseq2()

In [ ]:
params = sim_1c_data["params"]
params.index = params["id"]
params = params.join(dds.varm["LFC"])
params["A_est"] = np.sqrt(params["inphase"] ** 2 + params["outphase"] **2)
params = params.loc[np.logical_and((params["A"] > 0), (~np.isnan(params["A_est"])))]

In [ ]:
rho = scipy.stats.spearmanrho(params["A"], params["A_est"])[0]
ax = params.plot.scatter("A", "A_est")
ax.axline((0, 0), (1,np.log(2)), color='r')
ax.set_aspect("equal")
ax.text(0.8, 0.9, rf"$\rho$={rho:1.2f}", transform=ax.transAxes)
mp.ylim([0, 5])

### RNA-seq data under 2 conditions

In [ ]:
sim_2c_data = pf.generate_diffrhythmic_rnaseq(reps=3, n_genes=10000, rhy_frac=0.25, seed=seed+1)
sim_2c_data["exp_design"]["inphase"] = np.cos(2*np.pi/24*sim_2c_data["exp_design"].time.values)
sim_2c_data["exp_design"]["outphase"] = np.sin(2*np.pi/24*sim_2c_data["exp_design"].time.values)

In [ ]:
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(counts = sim_2c_data["counts"].T, metadata=sim_2c_data["exp_design"], design="~group + group:inphase + group:outphase", refit_cooks=True, inference=inference);
dds.deseq2()

In [ ]:
params = sim_2c_data["params"]
params.index = params["id"]
params = params.join(dds.varm["LFC"])
params["A_ctrl_est"] = np.sqrt(params["group[ctrl]:inphase"] ** 2 + params["group[ctrl]:outphase"] **2)
params["A_expt_est"] = np.sqrt(params["group[expt]:inphase"] ** 2 + params["group[expt]:outphase"] **2)

In [ ]:
fig, ax = mp.subplots(nrows=1, ncols=3, sharex="none",
                      sharey="none", figsize = (12, 4))
rho_de = scipy.stats.spearmanrho(params["DE_effect"], params["group[T.expt]"], nan_policy="omit")[0]
params.loc[~np.isnan(params["DE_effect"])].plot.scatter("DE_effect", "group[T.expt]", ax=ax[0])
ax[0].axline((0, 0), (1, np.log(2)), color='r')
ax[0].text(0.8, 0.1, rf"$\rho$={rho_de:1.2f}", transform=ax[0].transAxes)

rho_ctrl = scipy.stats.spearmanrho(params["A_ctrl"], params["A_ctrl_est"], nan_policy="omit")[0]
params.loc[~np.isnan(params["A_ctrl"])].plot.scatter("A_ctrl", "A_ctrl_est", ax=ax[1])
ax[1].axline((0, 0), (1, np.log(2)), color='r')
ax[1].text(0.8, 0.9, rf"$\rho$={rho_ctrl:1.2f}", transform=ax[1].transAxes)
ax[1].set_ylim([0, 5])

rho_expt = scipy.stats.spearmanrho(params["A_expt"], params["A_expt_est"], nan_policy="omit")[0]
params.loc[~np.isnan(params["A_expt"])].plot.scatter("A_expt", "A_expt_est", ax=ax[2])
ax[2].text(0.8, 0.9, rf"$\rho$={rho_expt:1.2f}", transform=ax[2].transAxes)
ax[2].axline((0, 0), (1, np.log(2)), color='r')
ax[2].set_ylim([0, 5])
fig.tight_layout()